# Lesson 01 Lab — Pruning Objectives, Constraints, and Delivery Boundaries

**Puzzle:** If half the weights become zero, has a mobile deployment objective been achieved?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Pruning is an engineering change with several possible objectives: package size, resident memory, first-token or first-frame latency, steady-state throughput, energy, and hardware cost. A sparsity percentage answers none of them by itself. The first deliverable is therefore a target card that connects one model revision and workload to a measurable deployment gate and a rollback condition.


## 0. Predict before running

1. Predict whether a masked 50% sparse matrix will materially beat its dense copy in ordinary dense GEMM.
2. Predict how the narrower layer changes parameters, FLOPs, and output shape.
3. Write one acceptance gate and one rollback gate for a latency-driven project.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The concrete objects are a dense linear layer, a same-shape masked layer, a physically narrower layer, their parameter tensors, the runtime input shape, and a latency distribution. The mask changes values; the narrower layer changes dimensions and the amount of dense work presented to the library.

- Logical zeros do not imply fewer dense instructions.
- A physical dimension change is visible to both the graph and the runtime.
- The acceptance metric must match the deployment objective and workload.


## 2. Derive the mechanism

For a dense matrix multiplication, leading work is approximately `2MKN`. Replacing half of W with zeros leaves M, N, and K unchanged when the operator still dispatches a dense GEMM. Physically reducing the output width changes N and therefore both arithmetic and output storage. A valid target card distinguishes logical sparsity, serialized representation, physical shape, kernel path, and end-to-end metric. This is why a parameter-count goal and an 80 ms first-frame SLO are related but not interchangeable.

### Mechanism at a glance

```mermaid
flowchart LR
  V["Value state<br/>which entries are zero?"] --> R["Representation state<br/>what is stored?"]
  R --> S["Shape state<br/>which axes changed?"]
  S --> E["Execution state<br/>which operator ran?"]
  E --> P["Product metric<br/>did the target improve?"]
```

### Walk it step by step

1. **Name the product objective.** Choose package size, memory, latency, throughput, energy, or cost and attach a measurable gate.
2. **Locate the structural change.** Distinguish zeros in a tensor from a changed tensor shape or stored representation.
3. **Locate the execution change.** Confirm whether the runtime dispatched a smaller dense operator or a supported sparse operator.
4. **Accept on the original objective.** A candidate succeeds only when quality and the named deployment metric both pass.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 1
LESSON_TITLE = 'Pruning Objectives, Constraints, and Delivery Boundaries'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260809
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | dense BF16 linear layer at the original shape |
| Candidate | 50% same-shape masking and a 50% physically narrower dense layer |
| Held constant | GPU, input batch, input width, dtype, warm-up, repetitions, and random seed |
| Measurements | logical sparsity, physical parameters, median/p95 latency, and output width |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare dense, same-shape masked, and physically narrower CUDA linear layers under one timing protocol.


## 5. Read the experiment code

The lab constructs all three candidates from one base weight tensor. The masked candidate preserves the dense shape, while the narrow candidate copies a selected subset of rows. CUDA events bracket only repeated forward calls after warm-up. Reading these rows together shows which optimization changed values and which changed the work exposed to the runtime.

Do not execute until the code implements the frozen table above.


In [2]:
dtype = torch.bfloat16
batch, in_features, out_features = 32, 2048, 2048
x = torch.randn(batch, in_features, device=DEVICE, dtype=dtype)
dense = nn.Linear(in_features, out_features, bias=False, device=DEVICE, dtype=dtype).eval()
masked = copy.deepcopy(dense)
mask = magnitude_mask(masked.weight, 0.50)
with torch.no_grad():
    masked.weight.mul_(mask)
keep = torch.arange(out_features // 2, device=DEVICE)
narrow = nn.Linear(in_features, keep.numel(), bias=False, device=DEVICE, dtype=dtype).eval()
with torch.no_grad():
    narrow.weight.copy_(dense.weight[keep])

dense_t = timing_summary(cuda_times(lambda: dense(x)))
masked_t = timing_summary(cuda_times(lambda: masked(x)))
narrow_t = timing_summary(cuda_times(lambda: narrow(x)))
metrics = {
    "dense_parameters": count_params(dense),
    "masked_parameters": count_params(masked),
    "masked_sparsity": zero_fraction(masked.weight),
    "narrow_parameters": count_params(narrow),
    "dense_output_width": out_features,
    "narrow_output_width": int(keep.numel()),
    "dense_median_ms": dense_t["median_ms"],
    "dense_p95_ms": dense_t["p95_ms"],
    "masked_median_ms": masked_t["median_ms"],
    "masked_p95_ms": masked_t["p95_ms"],
    "narrow_median_ms": narrow_t["median_ms"],
    "narrow_p95_ms": narrow_t["p95_ms"],
    "samples": {"dense": dense_t["samples_ms"], "masked": masked_t["samples_ms"], "narrow": narrow_t["samples_ms"]},
}
analysis = (
    f"The mask created {metrics['masked_sparsity']:.1%} logical sparsity but kept "
    f"{metrics['masked_parameters']:,} dense parameters and a {out_features}-wide output. "
    f"Its median was {metrics['masked_median_ms']:.6f} ms versus {metrics['dense_median_ms']:.6f} ms. "
    f"The physical candidate reduced parameters to {metrics['narrow_parameters']:,}, output width to "
    f"{keep.numel()}, and measured {metrics['narrow_median_ms']:.6f} ms. These numbers answer this CUDA "
    "operator workload only; they do not establish mobile first-frame latency."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Dense parameters | 4,194,304 |
| Masked logical sparsity | 50.00% |
| Narrow parameters | 2,097,152 |
| Dense median | 0.020192 ms |
| Masked median | 0.020112 ms |
| Narrow median | 0.019808 ms |


## 7. Interpret rather than merely print

The mask created 50.0% logical sparsity but kept 4,194,304 dense parameters and a 2048-wide output. Its median was 0.020112 ms versus 0.020192 ms. The physical candidate reduced parameters to 2,097,152, output width to 1024, and measured 0.019808 ms. These numbers answer this CUDA operator workload only; they do not establish mobile first-frame latency.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 1,
    "title": 'Pruning Objectives, Constraints, and Delivery Boundaries',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Zero weights satisfy a sparsity statistic; only a supported representation and a measured deployment path satisfy a performance objective.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 1,
  "title": "Pruning Objectives, Constraints, and Delivery Boundaries",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260809
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "dense_parameters": 4194304,
    "masked_parameters": 4194304,
    "masked_sparsity": 0.5,
    "narrow_parameters": 2097152,
    "dense_output_width": 2048,
    "narrow_output_width": 1024,
    "dense_median_ms": 0.020192000083625317,
    "dense_p95_ms": 0.023624000046402215,
    "masked_median_ms": 0.02011200040578842,
    "masked_p95_ms": 0.021137600764632225,
    "narrow_median_ms": 0.019807999953627586,
    "narrow_p95_ms": 0.021632000338286158,
    "samples": {
      "dense": [
        0.032735999673604965,
        0.023744000121951103,
        0.022112000733613968,
        0.022943999618291855,
        0.020416000857949257,
        0.02137600071728229

## 9. Make the bounded decision

> Zero weights satisfy a sparsity statistic; only a supported representation and a measured deployment path satisfy a performance objective.

**Acceptance/rollback:** Accept a pruning route only if the deployment metric improves under the frozen workload and the quality gate passes; otherwise keep the dense revision as the explicit rollback.

**Failure analysis:** A narrow microbenchmark can still mislead if first-frame setup, preprocessing, memory allocation, or a mobile runtime dominates. Conversely, masking may compress well on disk even when it does not accelerate the measured dense operator. Never transfer one objective's success to another objective without new evidence.


## 10. Extend the evidence

Add model serialization and a real deployment runtime, then repeat cold-start and steady-state tests separately. Record the graph dimensions and operator trace beside the latency table.

The full evidence boundary and references are in [`README.md`](README.md).
